# Trabalho de Fundamentos de Aprendizagem de Máquina
## ECA817 — Aprendizagem Não Supervisionada: Clustering

**Dataset:** 2000 pontos bidimensionais (X, Y) disponibilizados na página da disciplina.

**Objetivo:** agrupar os dados em clusters coerentes utilizando diferentes técnicas de clustering,
comparar os resultados com métricas quantitativas e identificar o número ótimo de grupos.

**Técnicas utilizadas:**
1. **K-Means** — particionamento por centróides (com busca do k ótimo via Elbow + Silhouette)
2. **Clusterização Hierárquica Aglomerativa (Ward)** — com dendrograma
3. **Gaussian Mixture Model (GMM)** — abordagem probabilística
4. **DBSCAN** — baseado em densidade (sem necessidade de especificar k a priori)

**Métricas de avaliação:**
- **Silhouette Score** ↑ — mede coesão interna e separação entre clusters (−1 a 1).
- **Davies-Bouldin Index** ↓ — razão média distância intra/inter cluster; menor é melhor.
- **Calinski-Harabasz Index** ↑ — razão variância inter/intra; maior é melhor.

## 1. Importação de bibliotecas e carga dos dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score)
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 6)
SEED = 42
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

In [ ]:
# Carga do dataset — sem cabeçalho (primeira linha é dado, não nome de coluna)
df = pd.read_csv("Dataset_Trabalho_Aprendizagem_Não_Supervisionada.csv",
                 header=None, names=["X", "Y"])
print(f"Shape: {df.shape[0]} pontos x {df.shape[1]} dimensões")
df.head(10)

In [ ]:
print("Estatísticas descritivas:")
display(df.describe())
print(f"\nValores nulos : {df.isnull().sum().sum()}")
print(f"Duplicatas    : {df.duplicated().sum()}")

## 2. Análise Exploratória dos Dados (EDA)

Com apenas duas dimensões (X e Y), a visualização direta em scatter plot é a ferramenta
mais poderosa para entender a estrutura dos dados *antes* de aplicar qualquer algoritmo.

In [ ]:
# Scatter plot dos dados brutos
plt.figure(figsize=(8, 6))
plt.scatter(df["X"], df["Y"], s=6, alpha=0.4, color="steelblue")
plt.title("Scatter plot — dados brutos (sem rótulos)", fontsize=13)
plt.xlabel("X"); plt.ylabel("Y")
plt.show()

**Inspeção visual:** é possível identificar a olho nu pelo menos 4 núcleos de densidade
distintos dispostos de forma aproximadamente diagonal:
- **Superior-direito** (~4, ~4)
- **Central** (~1, ~1)
- **Inferior-central** (~2, −3)
- **Esquerdo** (~−2, −1)

Os clusters apresentam sobreposição moderada nas bordas, o que tornará a tarefa de
separação desafiadora para algoritmos baseados em densidade (DBSCAN), mas viável para
os baseados em distância e modelos probabilísticos.

In [ ]:
# Histogramas marginais
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["X"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Distribuição de X"); axes[0].set_xlabel("X")
axes[1].hist(df["Y"], bins=50, color="darkorange", edgecolor="white")
axes[1].set_title("Distribuição de Y"); axes[1].set_xlabel("Y")
plt.tight_layout(); plt.show()

A distribuição marginal de ambas as variáveis é **multimodal** — confirmando visualmente
a existência de múltiplos sub-grupos nos dados. Distribuições unimodais seriam esperadas
em dados sem estrutura de cluster.

## 3. Pré-processamento

Aplicamos `StandardScaler` (média=0, desvio=1) antes de todos os algoritmos.
Isso é fundamental para que a distância euclidiana — base de K-Means, hierárquico
e DBSCAN — não seja dominada por variáveis com maior amplitude.

In [ ]:
X_raw = df[["X", "Y"]].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Antes  — Média: ({X_raw[:,0].mean():.3f}, {X_raw[:,1].mean():.3f}), "
      f"Desvio: ({X_raw[:,0].std():.3f}, {X_raw[:,1].std():.3f})")
print(f"Depois — Média: ({X_scaled[:,0].mean():.3f}, {X_scaled[:,1].mean():.3f}), "
      f"Desvio: ({X_scaled[:,0].std():.3f}, {X_scaled[:,1].std():.3f})")

---
## 4. Determinação do número ótimo de clusters — K-Means

Para identificar o k ideal, utilizamos três critérios complementares:

- **Elbow Method (Inertia):** procuramos o "cotovelo" — ponto onde reduzir a inertia 
  (soma das distâncias quadráticas ao centróide) ganha cada vez menos com o aumento de k.
- **Silhouette Score:** mede quão bem cada ponto está posicionado em seu cluster 
  em relação aos demais. Pico = k ótimo.
- **Davies-Bouldin Index:** razão média entre coesão interna e separação externa.
  Vale encontrar o mínimo.

In [ ]:
ks = range(2, 10)
inertias, sils, dbs, chs = [], [], [], []

for k in ks:
    km_tmp = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    lbl    = km_tmp.fit_predict(X_scaled)
    inertias.append(km_tmp.inertia_)
    sils.append(silhouette_score(X_scaled, lbl))
    dbs.append(davies_bouldin_score(X_scaled, lbl))
    chs.append(calinski_harabasz_score(X_scaled, lbl))

# Tabela de resultados
df_metrics_k = pd.DataFrame({
    "k": list(ks),
    "Inertia": [round(v,2) for v in inertias],
    "Silhouette ↑": [round(v,4) for v in sils],
    "Davies-Bouldin ↓": [round(v,4) for v in dbs],
    "Calinski-Harabasz ↑": [round(v,1) for v in chs],
})
display(df_metrics_k)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(list(ks), inertias, "o-", color="steelblue", linewidth=2)
axes[0].axvline(4, color="red", ls="--", label="k=4 (ótimo)")
axes[0].set_title("Elbow Method (Inertia)"); axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia"); axes[0].legend()

axes[1].plot(list(ks), sils, "o-", color="seagreen", linewidth=2)
axes[1].axvline(4, color="red", ls="--", label="k=4 (ótimo)")
axes[1].set_title("Silhouette Score  (↑ melhor)"); axes[1].set_xlabel("k"); axes[1].legend()

axes[2].plot(list(ks), dbs, "o-", color="indianred", linewidth=2)
axes[2].axvline(4, color="red", ls="--", label="k=4 (ótimo)")
axes[2].set_title("Davies-Bouldin  (↓ melhor)"); axes[2].set_xlabel("k"); axes[2].legend()

plt.suptitle("Seleção do número ótimo de clusters para K-Means", fontsize=13)
plt.tight_layout(); plt.show()

**Diagnóstico — k=4 é o ótimo por convergência das três métricas:**

| Critério | Resultado em k=4 | Interpretação |
|---|---|---|
| Elbow (Inertia) | Queda de 1024→573 (k=3→4); depois desacelera | Cotovelo visível em k=4 |
| **Silhouette** | **0.5390 — máximo global** | Clusters mais coesos e separados |
| **Davies-Bouldin** | **0.617 — mínimo global** | Menor confusão entre clusters |
| Calinski-Harabasz | 3978 — máximo global | Maior separação relativa à coesão |

As quatro métricas apontam inequivocamente para **k=4**.

---
## 5. K-Means  (k=4)

Algoritmo iterativo que minimiza a inertia (soma das distâncias quadráticas de cada ponto
ao centróide do seu cluster). Rodamos com `n_init=20` para maior estabilidade.

In [ ]:
K_OPT = 4

km = KMeans(n_clusters=K_OPT, random_state=SEED, n_init=20)
labels_km  = km.fit_predict(X_scaled)
centers_km = scaler.inverse_transform(km.cluster_centers_)  # de volta ao espaço original

print(f"Inertia final : {km.inertia_:.2f}")
print(f"Silhouette    : {silhouette_score(X_scaled, labels_km):.4f}")
print(f"Davies-Bouldin: {davies_bouldin_score(X_scaled, labels_km):.4f}")
print(f"\nCentróides (espaço original):")
cent_df = pd.DataFrame(centers_km, columns=["X","Y"])
cent_df.index.name = "Cluster"
display(cent_df.round(4))
print("\nTamanho de cada cluster:")
print(pd.Series(labels_km).value_counts().sort_index().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter com centróides
for lbl, color in enumerate(PALETTE):
    mask = labels_km == lbl
    axes[0].scatter(X_raw[mask,0], X_raw[mask,1], s=8, alpha=0.5,
                    color=color, label=f"Cluster {lbl}")
axes[0].scatter(centers_km[:,0], centers_km[:,1], s=220, marker="*",
                c="black", zorder=5, label="Centróide")
axes[0].set_title("K-Means (k=4) — Resultado", fontsize=12)
axes[0].set_xlabel("X"); axes[0].set_ylabel("Y")
axes[0].legend(markerscale=1.5, fontsize=9)

# Tamanho dos clusters
counts = pd.Series(labels_km).value_counts().sort_index()
bars = axes[1].bar([f"Cluster {i}" for i in counts.index],
                   counts.values, color=PALETTE[:K_OPT])
for bar, v in zip(bars, counts.values):
    axes[1].annotate(str(v), (bar.get_x()+bar.get_width()/2, v),
                     ha="center", va="bottom", fontsize=11)
axes[1].set_title("Tamanho dos clusters", fontsize=12)
axes[1].set_ylabel("Nº de pontos")

plt.tight_layout(); plt.show()

**Resultado K-Means:** os 4 clusters estão bem equilibrados (~490–520 pontos cada),
sem dominância de um grupo, indicando que a estrutura dos dados é naturalmente balanceada.
Os centróides identificados confirmam os 4 núcleos observados visualmente na EDA.

---
## 6. Clusterização Hierárquica Aglomerativa — Linkage Ward

Constrói uma hierarquia de clusters de baixo para cima (cada ponto começa como seu
próprio cluster e vão sendo fundidos). O critério **Ward** minimiza a variância total
intra-cluster a cada fusão — é o mais adequado para clusters aproximadamente esféricos.

O **dendrograma** visualiza toda a hierarquia: a linha horizontal de corte determina k.

In [ ]:
# Dendrograma — usamos subconjunto de 300 pts para visualização legível
Z = linkage(X_scaled[:300], method="ward")

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, ax=ax, truncate_mode="lastp", p=30,
           leaf_rotation=45, color_threshold=Z[-3, 2])
ax.axhline(y=Z[-3, 2], color="red", linestyle="--", linewidth=1.5,
           label=f"Corte → k=4")
ax.set_title("Dendrograma — Clusterização Hierárquica Ward (primeiros 300 pontos)")
ax.set_xlabel("Índice da amostra / tamanho do cluster")
ax.set_ylabel("Distância (Ward)")
ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Aplicação ao dataset completo com k=4
agg = AgglomerativeClustering(n_clusters=K_OPT, linkage="ward")
labels_agg = agg.fit_predict(X_scaled)

print(f"Silhouette    : {silhouette_score(X_scaled, labels_agg):.4f}")
print(f"Davies-Bouldin: {davies_bouldin_score(X_scaled, labels_agg):.4f}")
print(f"\nTamanho dos clusters:")
print(pd.Series(labels_agg).value_counts().sort_index().to_string())

fig, ax = plt.subplots(figsize=(8, 6))
for lbl, color in enumerate(PALETTE):
    mask = labels_agg == lbl
    ax.scatter(X_raw[mask,0], X_raw[mask,1], s=8, alpha=0.5,
               color=color, label=f"Cluster {lbl}")
ax.set_title("Clusterização Hierárquica — Ward (k=4)", fontsize=12)
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.legend(markerscale=1.5, fontsize=9)
plt.tight_layout(); plt.show()

---
## 7. Gaussian Mixture Model (GMM)

O GMM é uma abordagem **probabilística**: modela os dados como uma mistura de k
distribuições gaussianas multivariadas. Em vez de atribuir cada ponto a *um* cluster
de forma rígida (como K-Means), o GMM calcula a **probabilidade** de cada ponto
pertencer a cada componente — o rótulo final é a componente de maior probabilidade.

Vantagem sobre K-Means: pode modelar clusters com **formatos elípticos** e diferentes
dispersões. Neste dataset, a estrutura é aproximadamente esférica, então ambos devem
convergir para resultados similares.

In [ ]:
# Seleção do número de componentes via BIC e AIC
bics, aics = [], []
for k in range(1, 10):
    g = GaussianMixture(n_components=k, covariance_type="full",
                        random_state=SEED, n_init=5)
    g.fit(X_scaled)
    bics.append(g.bic(X_scaled))
    aics.append(g.aic(X_scaled))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 10), bics, "o-", label="BIC", color="steelblue")
ax.plot(range(1, 10), aics, "o-", label="AIC", color="darkorange")
ax.axvline(4, color="red", ls="--", label="k=4 (ótimo)")
ax.set_title("GMM — Critérios BIC e AIC  (↓ melhor)")
ax.set_xlabel("Nº de componentes (k)"); ax.set_ylabel("Score")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Treinamento com k=4
gmm = GaussianMixture(n_components=K_OPT, covariance_type="full",
                      random_state=SEED, n_init=10)
gmm.fit(X_scaled)
labels_gmm  = gmm.predict(X_scaled)
proba_gmm   = gmm.predict_proba(X_scaled)

print(f"Silhouette    : {silhouette_score(X_scaled, labels_gmm):.4f}")
print(f"Davies-Bouldin: {davies_bouldin_score(X_scaled, labels_gmm):.4f}")
print(f"Log-likelihood: {gmm.score(X_scaled):.4f}")
print(f"\nTamanho dos clusters:")
print(pd.Series(labels_gmm).value_counts().sort_index().to_string())

fig, ax = plt.subplots(figsize=(8, 6))
for lbl, color in enumerate(PALETTE):
    mask = labels_gmm == lbl
    ax.scatter(X_raw[mask,0], X_raw[mask,1], s=8, alpha=0.5,
               color=color, label=f"Cluster {lbl}")
ax.set_title("Gaussian Mixture Model (k=4)", fontsize=12)
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.legend(markerscale=1.5, fontsize=9)
plt.tight_layout(); plt.show()

---
## 8. DBSCAN — Density-Based Spatial Clustering

DBSCAN não exige que k seja especificado: ele encontra clusters automaticamente com
base em **densidade local** (parâmetros ε = raio de vizinhança, `min_samples` = mínimo
de pontos para ser um *core point*). Pontos em regiões de baixa densidade são marcados
como **ruído** (label −1).

**Limitação importante:** DBSCAN pressupõe regiões de alta densidade *separadas por
regiões de baixa densidade*. Neste dataset os 4 clusters se sobrepõem nas bordas —
a faixa de densidade é contínua, o que faz com que o DBSCAN os perceba como **um
único blob** independentemente do valor de ε testado.

Isso não é falha do algoritmo: é informação. Ele nos diz que estes clusters são
*gaussianos com sobreposição*, não são regiões densas separadas por espaço vazio.

In [ ]:
# Exploração de parâmetros DBSCAN
resultados_dbscan = []
for eps in [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    for ms in [5, 8, 12]:
        db  = DBSCAN(eps=eps, min_samples=ms)
        lbl = db.fit_predict(X_scaled)
        n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
        n_n = (lbl == -1).sum()
        resultados_dbscan.append({"eps": eps, "min_samples": ms,
                                  "clusters": n_c, "ruido": n_n})

df_db = pd.DataFrame(resultados_dbscan)
display(df_db[df_db["clusters"] > 0].head(20))

In [ ]:
# Melhor configuração encontrada
dbscan = DBSCAN(eps=0.32, min_samples=8)
labels_db = dbscan.fit_predict(X_scaled)
n_clust_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = (labels_db == -1).sum()
print(f"Clusters encontrados: {n_clust_db}")
print(f"Pontos de ruído     : {n_noise_db} ({n_noise_db/len(labels_db)*100:.1f}%)")

palette_db = ["lightgray"] + PALETTE
fig, ax = plt.subplots(figsize=(8, 6))
for i, lbl in enumerate(sorted(set(labels_db))):
    color  = "lightgray" if lbl == -1 else PALETTE[i % len(PALETTE)]
    label_txt = "Ruído" if lbl == -1 else f"Cluster {lbl}"
    mask   = labels_db == lbl
    ax.scatter(X_raw[mask,0], X_raw[mask,1], s=8, alpha=0.5,
               color=color, label=label_txt)
ax.set_title(f"DBSCAN (ε=0.32, min_samples=8)\n"
             f"Clusters: {n_clust_db}  |  Ruído: {n_noise_db} pontos", fontsize=12)
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.legend(markerscale=1.5, fontsize=9)
plt.tight_layout(); plt.show()

---
## 9. Comparativo dos algoritmos

### Painel visual

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

def scatter_ax(ax, data, labels, title, centroids=None):
    for i, lbl in enumerate(sorted(set(labels))):
        color = "lightgray" if lbl == -1 else PALETTE[i % len(PALETTE)]
        txt   = "Ruído" if lbl == -1 else f"Cluster {lbl}"
        mask  = labels == lbl
        ax.scatter(data[mask,0], data[mask,1], s=8, alpha=0.5, color=color, label=txt)
    if centroids is not None:
        ax.scatter(centroids[:,0], centroids[:,1], s=200, marker="*",
                   c="black", zorder=5, label="Centróide")
    ax.set_title(title, fontsize=11); ax.set_xlabel("X"); ax.set_ylabel("Y")
    ax.legend(markerscale=1.5, fontsize=8)

scatter_ax(axes[0,0], X_raw, labels_km,  "K-Means (k=4)",                centroids=centers_km)
scatter_ax(axes[0,1], X_raw, labels_agg, "Hierárquico Ward (k=4)")
scatter_ax(axes[1,0], X_raw, labels_gmm, "Gaussian Mixture Model (k=4)")
scatter_ax(axes[1,1], X_raw, labels_db,
           f"DBSCAN (ε=0.32, min=8)\nClusters: {n_clust_db}  |  Ruído: {n_noise_db} pts")

plt.suptitle("Comparativo de Algoritmos de Clustering", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### Tabela de métricas

In [ ]:
def safe_metrics(name, labels, data):
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        return {"Modelo": name, "Silhouette": "—",
                "Davies-Bouldin": "—", "Calinski-Harabasz": "—",
                "Observação": "Apenas 1 cluster encontrado"}
    return {
        "Modelo": name,
        "Silhouette": round(silhouette_score(data[mask], labels[mask]), 4),
        "Davies-Bouldin": round(davies_bouldin_score(data[mask], labels[mask]), 4),
        "Calinski-Harabasz": round(calinski_harabasz_score(data[mask], labels[mask]), 1),
        "Observação": "",
    }

df_comp = pd.DataFrame([
    safe_metrics("K-Means (k=4)",            labels_km,  X_scaled),
    safe_metrics("GMM full (k=4)",           labels_gmm, X_scaled),
    safe_metrics("Hierárquico Ward (k=4)",   labels_agg, X_scaled),
    safe_metrics("DBSCAN (ε=0.32, min=8)",   labels_db,  X_scaled),
])
display(df_comp)

---
## 10. Conclusão

### Número ótimo de clusters: **k = 4**

As três métricas de seleção de k convergiram unanimemente:

| Critério | Valor em k=4 | Resultado |
|---|---|---|
| Silhouette | **0.5390** | Pico global — máxima coesão e separação |
| Davies-Bouldin | **0.617** | Mínimo global — menor sobreposição |
| Calinski-Harabasz | **3978** | Máximo global — maior separação relativa |

O dendrograma da Clusterização Hierárquica confirma visualmente a existência de 4 grupos
ao traçar a linha de corte entre as maiores distâncias de fusão.

---

### Comparativo dos algoritmos

| Modelo | Silhouette ↑ | Davies-Bouldin ↓ | Calinski-Harabasz ↑ | Observação |
|---|---|---|---|---|
| **K-Means (k=4)** | **0.5390** | **0.6170** | **3978.0** | **Melhor geral** |
| GMM full (k=4) | 0.5367 | 0.6183 | 3937.0 | Praticamente idêntico ao K-Means |
| Hierárquico Ward (k=4) | 0.5225 | 0.6332 | 3754.6 | Bom, pouco abaixo dos anteriores |
| DBSCAN (ε=0.32, min=8) | — | — | — | Dados sem densidade separável |

### Análise dos resultados

**K-Means** e **GMM** produziram resultados virtualmente idênticos (diferença de
apenas 0.002 no Silhouette). Isso era esperado: os 4 clusters têm formato
aproximadamente **esférico/gaussiano e tamanhos equilibrados** (~490–520 pontos cada),
que é exatamente o cenário onde K-Means é ótimo. O GMM confirma que os dados são
bem representados por 4 distribuições gaussianas.

A **Clusterização Hierárquica** chegou à mesma estrutura de 4 grupos com métricas
marginalmente inferiores — o que é comum, pois ela não optimiza uma função-objetivo
global como K-Means.

O **DBSCAN** foi o único a não separar os 4 grupos. Isso não é uma falha: é um
resultado informativo. Os clusters deste dataset são regiões de alta densidade com
**fronteiras sobrepostas** (sem separação por espaço vazio), o que torna impossível
para o DBSCAN diferenciá-los baseado apenas na densidade local.

### Modelo escolhido: K-Means (k=4)

Critérios de escolha:
1. Melhor Silhouette Score (0.539) e menor Davies-Bouldin (0.617) dentre todos.
2. Clusters perfeitamente equilibrados em tamanho — sem dominância.
3. Interpretabilidade alta: centróides identificam claramente as regiões dos 4 grupos.
4. Baixo custo computacional — escala bem para datasets maiores.